# H2A variables calculation: reproduce discrete variables calculation

This notebook reproduces the 'H2A_stats_input_51subs.csv' file, which contains the discrete variables calculated from the H2A_vars_calc.ipynb notebook. However, rather than calculating the discrete vars from the pickle files which, in turn were calculated from V3D processed data, this nb calculates from the knt files. The aim is to make sure that prospective users will be able to reproduce the results of the paper by downloading the knt files and running this notebook.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import seaborn as sns
import plotly.express as px
import pickle
from fnmatch import fnmatch

import sys, os
sys.path.insert(1, r'./../functions')  # add to pythonpath

from critic_damp import critic_damp
from detecta import detect_onset
from simila import similarity

# scipy and numpy have too many future warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
#@title #### Plot style {display-mode: "form"}
sns.set_context('notebook', font_scale=1, rc={"lines.linewidth": 1})
sns.set_style('whitegrid')
colors = sns.color_palette()
display(colors)

pd.set_option('display.precision', 3)

## File directories
The compressed files contained in the WBDSascii.zip and WBDSinfo.xlsx should be dowloaded and located in the same directory.

In [ ]:
fileDir     = Path(r'C:\Users\Reginaldo\Downloads\WBDSascii\51subjs')
fileDir_c3d = Path(r'C:\Users\Reginaldo\Downloads\WBDSc3d\WBDSc3d')
path2       = Path(r'./../data')

## Metadata file
This file contains 51 subjects, 42 subjects from the previous published dataset and 9 new subjects.

In [ ]:
# Read the CSV file into a DataFrame
filename = path2 / 'WBDSinfo.csv'
df_x = pd.read_csv(filename, index_col=False)
df_x

In [ ]:
# Drop duplicates
info = df_x.drop_duplicates(subset='Subject', inplace=False)
info.iloc[:, :9]

Number of subjects by age group and gender

In [ ]:
display(info[['Subject', 'AgeGroup', 'Gender']].groupby(['AgeGroup', 'Gender']).count())
N = len(info['Subject'])
Ns = {age: info[info['AgeGroup']==age]['Subject'].nunique() for age in info['AgeGroup'].unique()}
print(str(N) + ' participants')

## Parameters for analysis

In [ ]:
conditions = ['V1','V2','V3','V4','V5','V6'] # speed conditions
directions  = ['X','Y','Z']
joints     = ['Pelvis', 'Hip','Knee','Ankle']

## Import pickle files
These pickle files in the WBDS_process_V3D_files.ipynb, after the data being cleaned by similarity function.

In [ ]:
# # load young data
# with open(path2 / 'young_dataM.pickle', 'rb') as y_file:
#     young_data = pickle.load(y_file)

# # Load older data
# with open(path2 / 'older_dataM.pickle', 'rb') as o_file:
#     older_data = pickle.load(o_file)

## Import time series data
### From the updated data, including the new subjects (43 to 51). 

In [ ]:
if True:
    # Exclude Pelvis joint from the analysis, as it is not a lower limb joint
    joints = [j for j in joints if j != 'Pelvis']
    # From txt files posted in Figshare
    suppMom= np.empty(shape=(101, N, len(conditions)))
    Mom    = np.empty(shape=(101, N, len(joints), len(conditions)))
    Ang    = np.empty(shape=(101, N, len(joints), len(conditions)))
    grf    = np.empty(shape=(101, N, len(joints), len(conditions)))
    Pow    = np.empty(shape=(101, N, len(joints), len(conditions)))
    for s, subject in enumerate(info['Subject']):
        idx = "%02i"% (subject)
        for v, velocity in enumerate(conditions):
            idy = "%02i"% (v+1)
            filename = os.path.join(
                fileDir, 'WBDS' + idx + 'walkT' + idy + 'ang.txt'
            )
            WBDSa = pd.read_csv(filename, sep='\t', index_col=None, engine='c', encoding='utf-8')
            filename = os.path.join(fileDir, 'WBDS' + idx + 'walkT' + idy + 'knt.txt')
            WBDS = pd.read_csv(filename, sep='\t', index_col=None, engine='c', encoding='utf-8')
            # Support moment
            suppMom[: ,s ,v] = -WBDS['RHipMomentZ'].values + WBDS['RKneeMomentZ'].values + WBDS['RAnkleMomentZ'].values
            for j, joint in enumerate(joints):
                # Moment, GRF, Angle, Power
                Mom[:, s, j, v] = WBDS['R' + joint + 'MomentZ'].values
                grf[:, s, j, v] = WBDS['RGRF'+ directions[j]].values
                Ang[:, s, j, v] = WBDSa['R' + joint + 'AngleZ'].values
                Pow[:, s, j, v] = WBDS['R' + joint + 'Power'].values

### Enable the following cell to import the data from the pickle files. This is the same data that was used to calculate the discrete variables in the H2A_vars_calc.ipynb notebook. 

### From data in the local BMClab directory WBDS/C3D
* Reprocessed in Visual 3D software using Matlab *runPipelineWBDS2023.m* script
* Exported V3D data were then reprocessed using *WBDS_process_V3D_files.ipynb* and pickle files containing Python dictionaries were exported

In [ ]:
if False:
    simila = 'simila_1' # simila_0: uncleaned data; simila_1: cleaned data

    # load data
    with open(path2 / 'young_dataM.pickle', 'rb') as y_file:
        young_data = pickle.load(y_file)

    # Extract data from the young_data dictionary
    Ny = len(list(young_data.keys()))
    suppMomY = np.empty(shape=(101, Ny, len(conditions)))
    MomY    = np.empty(shape=(101, Ny, len(joints), len(conditions)))
    AngY    = np.empty(shape=(101, Ny, len(joints), len(conditions)))
    grfY   = np.empty(shape=(101, Ny, len(joints), len(conditions)))
    PowY    = np.empty(shape=(101, Ny, len(joints), len(conditions)))
    for s, subject in enumerate(list(young_data.keys())):
        for v, velocity in enumerate(conditions): 
            frsY   = young_data[subject][simila]['GRF_'+velocity]
            # Support moment
            hipY  = young_data[subject][simila]['HipMoment_'+velocity][:,2]
            kneeY = young_data[subject][simila]['KneeMoment_'+velocity][:,2]
            ankleY= young_data[subject][simila]['AnkleMoment_'+velocity][:,2]
            suppMomY[: ,s ,v] = -hipY + kneeY + ankleY
            for j, joint in enumerate(joints):
                angleY = young_data[subject][simila][joint+'_'+velocity]
                torqueY= young_data[subject][simila][joint+'Moment_'+velocity]
                powerY = young_data[subject][simila][joint+'Power_'+velocity]
                # Moment, GRF, Angle, Power
                MomY[:, s, j, v] = torqueY[:,2]
                grfY[:, s, j, v] = frsY[:,j]
                AngY[:, s, j, v] = angleY[:,2]
                PowY[:, s, j, v] = powerY[:,2]

    # Load older data
    o_file = open(path2 / 'older_dataM.pickle', 'rb')
    older_data = pickle.load(o_file)

    # Extract data from the older_data dictionary
    No = len(list(older_data.keys()))
    suppMomO= np.empty(shape=(101, No, len(conditions)))
    MomO    = np.empty(shape=(101, No, len(joints), len(conditions)))
    AngO    = np.empty(shape=(101, No, len(joints), len(conditions)))
    grfO    = np.empty(shape=(101, No, len(joints), len(conditions)))
    PowO    = np.empty(shape=(101, No, len(joints), len(conditions)))
    for s, subject in enumerate(list(older_data.keys())):
        for v, velocity in enumerate(conditions): 
            frsO   = older_data[subject][simila]['GRF_'+velocity]
            # Support moment
            hipO  = older_data[subject][simila]['HipMoment_'+velocity][:,2]
            kneeO = older_data[subject][simila]['KneeMoment_'+velocity][:,2]
            ankleO= older_data[subject][simila]['AnkleMoment_'+velocity][:,2]
            suppMomO[: ,s ,v] = -hipO + kneeO + ankleO
            for j, joint in enumerate(joints):
                angleO = older_data[subject][simila][joint+'_'+velocity]
                torqueO= older_data[subject][simila][joint+'Moment_'+velocity]
                powerO = older_data[subject][simila][joint+'Power_'+velocity]
                # Moment, GRF, Angle, Power
                MomO[:, s, j, v] = torqueO[:,2]
                grfO[:, s, j, v] = frsO[:,j]
                AngO[:, s, j, v] = angleO[:,2]
                PowO[:, s, j, v] = powerO[:,2]

    # Merge data from Young and Older Group
    Ang = np.concatenate((AngY,AngO), axis=1)
    Mom = np.concatenate((MomY,MomO), axis=1)
    Pow = np.concatenate((PowY,PowO), axis=1)
    grf = np.concatenate((grfY,grfO), axis=1)
    suppMom = np.concatenate((suppMomY,suppMomO), axis=1)

## Quality checking data

We will plot angles and GRF as these variables are directly measured by the motion capture and force platform systems. Moments and powers are, in turn, measures derived from kinematics and GRF.

### Create dataframe with time series

In [ ]:
# Create df with time series data
Ang2 = Ang.flatten('F')
Mom2 = Mom.flatten('F')
Pow2 = Pow.flatten('F')
grf2 = grf.flatten('F')
time = list(range(0,101)) * 51 * 3 * 6 # time column
# Repeat values of dataframe to fit time series data
df_rep = info.loc[info.index.repeat(101)].reset_index(drop=True) #101 times
df_rep = pd.concat([df_rep]*3, ignore_index=True) #3 times
df_rep = pd.concat([df_rep]*6, ignore_index=True) #6 times
df_rep.drop(labels='FileName', axis=1, inplace=True) # drop filename column
# Joints labels
joint_lbls = list(np.repeat(joints,Ang.shape[1] * Ang.shape[0]))
joint_lbls2 = joint_lbls * 6
# Speed labels
speed_lbl = list(np.repeat(conditions, len(joint_lbls)))
# Create df
data = np.vstack((Ang2, Mom2, Pow2, grf2)).T
dfall = pd.DataFrame(data=data, columns=['Angles','Moments','Powers','GRFs'])
# Add columns into df
dfall['joints']  = joint_lbls2
dfall['speed']   = speed_lbl
dfall['time']    = time
# Merge dfs
dfall = pd.concat([df_rep, dfall], axis=1)
dfall = dfall[['Subject', 'AgeGroup', 'Age', 'Height', 'Mass', 'Gender', 'Dominance', 
               'LegLength', 'Static1', 'Static2', 'GaitSpeed(m/s)', 'TreadHands', 
               'FP_RightFoot', 'FP_LeftFoot', 'Notes', 'BorgScale','joints','speed',
               'time','Angles','Moments','Powers','GRFs']] # reorder columns
dfall

### Joint angles, moments, powers and GRFs

In [ ]:
for age, n in Ns.items():
    fig1 = px.line(dfall[dfall['AgeGroup']==age], x='time', y='Angles', color='Subject', facet_row='joints',
                  facet_col='speed', hover_data=['Subject'], height=708, width=1024,
                  title=f'Group: {age} (N={n}) - Joint angles [deg]')
    fig1.for_each_yaxis(lambda y: y.update(autorange=True, matches=None))
    fig1.show()

In [ ]:
for age, n in Ns.items():
    fig2 = px.line(dfall[dfall['AgeGroup']==age], x='time', y='Moments', color='Subject', facet_row='joints',
                  facet_col='speed', hover_data=['Subject'], height=700, width=1024,
                  title=f'Group: {age} (N={n}) - Joint moments [Nm/kg]')
    fig2.for_each_yaxis(lambda y: y.update(autorange=True, matches=None))
    fig2.show()

In [ ]:
for age, n in Ns.items():
    fig3 = px.line(dfall[dfall['AgeGroup']==age], x='time', y='Powers', color='Subject', facet_row='joints',
                  facet_col='speed', hover_data=['Subject'], height=608, width=1024,
                  title=f'Group: {age} (N={n}) - Joint powers [W/kg]')
    fig3.for_each_yaxis(lambda y: y.update(autorange=True, matches=None))
    fig3.show()

In [ ]:
for age, n in Ns.items():
    fig4 = px.line(dfall[dfall['AgeGroup']==age], x='time', y='GRFs', color='Subject', facet_row='joints',
                  facet_col='speed', hover_data=['Subject'], height=708, width=1024,
                  title=f'Group: {age} (N={n}) - GRFs [N/kg]')
    fig4.for_each_yaxis(lambda y: y.update(autorange=True, matches=None))
    fig4.show()

In [ ]:
# Save dataframe
#dfall.to_csv(path2 / 'curves.csv')

## Spatiotemporal parameters

In [ ]:
freq_t = 150 #Kinematics frequency
freq_f = 300 #Kinetics frequency

In [ ]:
# Filter parameters
b_cd, a_cd, fc_cd = critic_damp(fcut=10, freq=freq_f, npass=2,
                                fcorr=False, filt='butter')

### Detect gait events and calculate step length and cadence
Obs: the Left force plate broke after the original WBDS (N=42 subjects) was published. So the remaining 9 subjects included in this analysis had only right leg external force data available. This issue lead us to use only the right side data for the the subjects (n=51).

In [ ]:
RstepTime2  = np.empty(shape=(N,len(conditions)))
Rcadence    = np.empty(shape=(N,len(conditions)))
Rcadence2    = np.empty(shape=(N,len(conditions)))
gaitSpeed2  = np.empty(shape=(N,len(conditions)))
RstepLength2= np.empty(shape=(N,len(conditions)))
RcycleTime2   = np.empty(shape=(N,len(conditions)))
RsuppTime2    = np.empty(shape=(N,len(conditions)))
RsuppTimePerc2= np.empty(shape=(N,len(conditions)))
beltDistTravel2=np.empty(shape=(N,len(conditions)))
for s, fname in enumerate(info['Subject']):
    idx = "%02i"% (fname)
    for v, velocity in enumerate(conditions):
        idy = "%02i"% (v+1)
        fname_t = os.path.join(fileDir,'WBDS' + idx + 'walkT' + idy + 'mkr.txt')
        fname_f = os.path.join(fileDir,'WBDS' + idx + 'walkT' + idy + 'grf.txt')

        df_t = pd.read_csv(fname_t, sep='\t', index_col=False, encoding='utf-8', engine='c')
        df_f = pd.read_csv(fname_f, sep='\t', index_col=False, encoding='utf-8', engine='c')

        time_t = np.linspace(0, len(df_t)/freq_t, num=len(df_t))
        time_f = np.linspace(0, len(df_f)/freq_f, num=len(df_f))

        # resample mkr data to the forces frequency
        R = np.interp(time_f, time_t, df_t['R.HeelX'].values)
        L = np.interp(time_f, time_t, df_t['L.HeelX'].values)

        # Filtering signal
        Fy1 = signal.filtfilt(b_cd, a_cd, df_f['Fy1'])
        Fy2 = signal.filtfilt(b_cd, a_cd, df_f['Fy2'])

        # Detect gait events in GRF signal
        threshold = 30
        n_above = int(0.1*freq_f)
        n_below = int(0.05*freq_f)
        threshold2 = 10*threshold  # N
        n_above2 = int(0.02*freq_f)
        idxR = detect_onset(Fy2, threshold, n_above, n_below,
                           threshold2, n_above2, show=False)

        # Conditional to avoid detecting wrong event (e.g. toeoff) or wrong phase (eg midstance instead of foot strike)
        if (R[idxR[0,0]] < L[idxR[0,0]]) or (Fy2[idxR[0,0]] > threshold*2.5):
            idxR = idxR[1:,:]

        # Gait Speed
        filenameC3D = 'WBDS' + idx + 'walkT' + idy + '.c3d'
        gaitSpeed = df_x.loc[df_x['FileName'] == filenameC3D, 
                             'GaitSpeed(m/s)'].values.astype(float)
        gaitSpeed2[s, v] = gaitSpeed[0]

        # Cycle time
        RcycleTimex = np.diff(idxR[:,0])/freq_f
        RcycleTime = np.median(RcycleTimex)
        RcycleTime2[s, v] = RcycleTime

        # Cadence stride
        # Because left FP is not working from subjects 43 onwards
        Rcadencex = np.median(120/RcycleTime)
        Rcadence[s, v] = Rcadencex

        # Stride Length
        beltDistTravel = gaitSpeed*RcycleTimex
        beltDistTravel2[s, v] = np.median(beltDistTravel)
        RstepLengthx = np.diff(R[idxR[:,0]])
        RstepLengthx = (RstepLengthx/1000) + beltDistTravel
        RstepLength = np.median(RstepLengthx)
        RstepLength2[s, v] = RstepLength

        # Support time
        RsuppTimex = np.diff(idxR, axis=1)/freq_f
        RsuppTime = np.median(RsuppTimex)
        RsuppTime2[s, v] = RsuppTime

        # converting support time to percentage of gait cycle
        RsuppTimePercx = (RsuppTime/RcycleTime)*100
        RsuppTimePerc2[s, v] = int(round(RsuppTimePercx))

## Dependent variables extraction
Calculate dependent variables according to the literature.

### DeVita and Hortobagyi (2000)

**Joint Impulses**
<br>
"Angular impulse by the extensor or plantar flexor muscles was calculated throughout stance from the support and ankle torques and during the initial half of stance from the knee and hip torques. Flexor angular impulse at the hip in late stance and the maximum plantar flexor torque at the ankle were also assessed."

**Joint Work**
<br>
Joint powers were calculated as the product of the joint moments and angular velocities. Work at each joint was
calculated as the area under portions of the power curves during extensor torque generation. Positive work indicates
that the observed joint torque generated mechanical energy and contributed to propelling the individual. Negative work
indicates that the observed joint torque absorbed mechanical energy. Positive work was calculated at the hip during the first half of stance, at the knee during midstance, and at the ankle during late stance. Negative work was calculated at the knee joint during early stance.

### Kuhman et al. (2018) Journal of Biomechanics
To create single metrics representing biomechanical plasticity, we created ratios of hip extensor to ankle plantarﬂexor peak torques, angular impulses, peak positive powers, and positive work. Increased ratio values indicate increased hip relative to ankle joint kinetics, representative of increased magnitudes of biomechanical plasticity.

### Peak Joint Moment, Joint Impulses and Joint Work

In [ ]:
timePow = np.empty(shape=(101,N,len(conditions)))
omega2  = np.empty(shape=(101,N,len(joints),len(conditions)))
jointPow2= np.empty(shape=(101,N,len(joints),len(conditions)))
posWork  = np.empty(shape=(N,len(joints),len(conditions)))
negWork  = np.empty(shape=(N,len(joints),len(conditions)))
posImp  = np.empty(shape=(N,len(joints),len(conditions)))
negImp  = np.empty(shape=(N,len(joints),len(conditions)))
posImp1st  = np.empty(shape=(N,len(joints),len(conditions)))
negImp1st  = np.empty(shape=(N,len(joints),len(conditions)))
suppMomposImp = np.empty(shape=(N,len(conditions)))
maxMom = np.empty(shape=(N,len(joints),len(conditions)))
minMom = np.empty(shape=(N,len(joints),len(conditions)))
for s, fname in enumerate(info['Subject']):
    for v, velocity in enumerate(conditions):
        timePow[:, s, v] = np.linspace(0,RcycleTime2[s, v],101)
        for j, joint in enumerate(joints):
            # joint power
            jointPow = Pow[:,s,j,v]
            # Joint power support phase
            fim = int(RsuppTimePerc2[s, v])
            jointPowSupp = jointPow[:fim]
            # Time vector based on suport phase duration
            tsupp = RsuppTime2[s, v]
            tv = np.linspace(0,tsupp,len(jointPowSupp))
            powpos = jointPowSupp[jointPowSupp > 0]
            powneg = jointPowSupp[jointPowSupp < 0]
            ipos = np.nonzero(jointPowSupp > 0)
            ineg = np.nonzero(jointPowSupp < 0)
            posWork[s, j, v] = np.trapz(jointPowSupp[ipos], x=tv[ipos])
            negWork[s, j, v] = np.trapz(jointPowSupp[ineg], x=tv[ineg])
            # Calculate joint impulses during stance
            xmom = Mom[:fim, s, j, v]
            xpos = xmom[xmom > 0]
            xneg = xmom[xmom < 0]
            ipos2= np.nonzero(xmom > 0)
            ineg2= np.nonzero(xmom < 0)
            posImp[s, j, v] = np.trapz(xmom[ipos2],x=tv[ipos2])
            negImp[s, j, v] = np.trapz(xmom[ineg2],x=tv[ineg2])
            # Joint impulse during first half of stance
            fim1st = round(fim/2)
            xmom2 = Mom[0:fim1st, s, j, v]
            tv1st = np.linspace(0,tsupp/2,len(xmom2))
            ipos1st= np.nonzero(xmom2 > 0)
            ineg1st= np.nonzero(xmom2 < 0)
            posImp1st[s, j, v] = np.trapz(xmom2[ipos1st],x=tv1st[ipos1st])
            negImp1st[s, j, v] = np.trapz(xmom2[ineg1st],x=tv1st[ineg1st])
            # Calc max and min Moment values
            maxMom[s, j, v] = np.max(Mom[:60, s, j, v], axis=0)
            minMom[s, j, v] = np.min(Mom[:60, s, j, v], axis=0)
            jointPow2[:, s, j, v] = jointPow

        # Support moment impulse
        suppMomStance = suppMom[:fim, s, v]
        ipos3= np.nonzero(suppMomStance > 0)
        suppMomposImp[s, v] = np.trapz(suppMomStance[ipos3],x=tv[ipos3])

In [ ]:
# Joint impulse
suppMomposImp2    = suppMomposImp[:, :len(conditions)]
ankleEXTimp       = posImp[:, 2, :len(conditions)]
kneeEXTimp        = posImp1st[:, 1, :len(conditions)]
hipEXTimp         = -negImp1st[:, 0, :len(conditions)]
hipFLXimp         = -posImp[:, 0, :len(conditions)]
hip2ankleRatioImp = hipEXTimp / ankleEXTimp

In [ ]:
# Peak moment values
hipPeak  = -minMom[:,0,:len(conditions)]
kneePeak = maxMom[:,1,:len(conditions)]
anklePeak= maxMom[:,2,:len(conditions)]
hip2ankleRatio = hipPeak/anklePeak

In [ ]:
hipPOSwork  = posWork[:,0,:len(conditions)]
hipNEGwork  = negWork[:,0,:len(conditions)]
kneePOSwork = posWork[:,1,:len(conditions)]
kneeNEGwork = negWork[:,1,:len(conditions)]
anklePOSwork= posWork[:,2,:len(conditions)]
ankleNEGwork= negWork[:,2,:len(conditions)]
hip2ankleRatioWork = hipPOSwork/anklePOSwork

## Create dataframe for statistics

In [ ]:
nrows = N*len(conditions)
RstepTime2  = np.reshape(RstepTime2, (nrows,))
Rcadence    = np.reshape(Rcadence, (nrows,))
gaitSpeed2  = np.reshape(gaitSpeed2, (nrows,))
RstepLength2= np.reshape(RstepLength2, (nrows,))

In [ ]:
# Create pandas dataFrame to store results
cols = ['Speed','StepTime','Cadence','StepLength']
data = np.array([gaitSpeed2,RstepTime2,Rcadence,RstepLength2])
df_stp2 = pd.DataFrame(data=data.T, columns=cols)
df_stp2['Subject'] = np.repeat(info['Subject'].tolist(), len(conditions))

In [ ]:
# Adding to the df
df_stp2['PeakHipMom']    = np.reshape(hipPeak,(nrows,))
df_stp2['PeakKneeMom']   = np.reshape(kneePeak,(nrows,))
df_stp2['PeakAnkleMom']  = np.reshape(anklePeak,(nrows,))
df_stp2['Hip2AnkleRatio']= np.reshape(hip2ankleRatio,(nrows,))
df_stp2['hipEXTimp']     = np.reshape(hipEXTimp,(nrows,))
df_stp2['hipFLXimp']     = np.reshape(hipFLXimp,(nrows,))
df_stp2['kneeEXTimp']    = np.reshape(kneeEXTimp,(nrows,))
df_stp2['ankleEXTimp']   = np.reshape(ankleEXTimp,(nrows,))
df_stp2['hip2ankleRatioImp']= np.reshape(hip2ankleRatioImp,(nrows,))
df_stp2['hipPOSwork']     = np.reshape(hipPOSwork,(nrows,))
df_stp2['hipNEGwork']     = np.reshape(hipNEGwork,(nrows,))
df_stp2['kneePOSwork']    = np.reshape(kneePOSwork,(nrows,))
df_stp2['kneeNEGwork']     = np.reshape(kneeNEGwork,(nrows,))
df_stp2['anklePOSwork']     = np.reshape(anklePOSwork,(nrows,))
df_stp2['ankleNEGwork']     = np.reshape(ankleNEGwork,(nrows,))
df_stp2['hip2ankleRatioWork']= np.reshape(hip2ankleRatioWork,(nrows,))

In [ ]:
df_stp2['AgeGroup'] = list(np.repeat(info['AgeGroup'].tolist(),
                                     len(conditions)))

In [ ]:
age       = np.repeat(info['Age'].values,len(conditions))
height    = np.repeat(info['Height'].values,len(conditions))
mass      = np.repeat(info['Mass'].values,len(conditions))
gender    = np.repeat(info['Gender'].values,len(conditions))
LegLength = np.repeat(info['LegLength'].values,len(conditions))

In [ ]:
df_stp2['Age'] = age
df_stp2['Height'] = height
df_stp2['Mass'] = mass
df_stp2['Gender'] = gender
df_stp2['LegLength'] = LegLength

In [ ]:
# Gait speed index
gSpeedLabel = np.tile(conditions,N)
df_stp2['SpeedCategory'] = gSpeedLabel

In [ ]:
# Froude gait speed
g = 9.81
df_stp2['SpeedRaw'] = df_stp2['Speed'].values*np.sqrt((g*df_stp2['LegLength'].values))

## Export df as csv file

In [ ]:
df = df_stp2
df.drop(columns=['SpeedRaw', 'StepTime'], inplace=True)  # unused columns wbdsRedist_clean3.csv
# Height cm to meters and stride to step length
df['Height'] = df['Height']/100
df['StepLength'] = df['StepLength']/2
# Append BMI and Froude number for gait speed
df = df.assign(BMI = df['Mass']/df['Height']**2)
df = df.assign(SpeedFroude = df['Speed'].values/np.sqrt((9.81*df['LegLength'].values)))
# rename columns
df.rename(columns={'SpeedCategory':'SpeedCat',
                   'PeakHipMom':'Hip_M', 'PeakKneeMom': 'Knee_M',
                   'PeakAnkleMom': 'Ankle_M', 'Hip2AnkleRatio': 'H2A_M',
                   'hipEXTimp': 'Hip_Iext', 'hipFLXimp': 'Hip_Iflx',
                   'kneeEXTimp': 'Knee_Iext', 'ankleEXTimp': 'Ankle_Iext',
                   'hip2ankleRatioImp': 'H2A_I', 'hipPOSwork': 'Hip_Wpos',
                   'hipNEGwork': 'Hip_Wneg', 'kneePOSwork': 'Knee_Wpos',
                   'kneeNEGwork': 'Knee_Wneg', 'anklePOSwork': 'Ankle_Wpos',
                   'ankleNEGwork': 'Ankle_Wneg', 'hip2ankleRatioWork': 'H2A_W'},
          inplace=True)
# comfortable speed as a new variable (column)
df = df.assign(SpeedComf = df.Speed)
for s in df['Subject'].unique():
    df.loc[df['Subject']==s, 'SpeedComf'] = df[(df['Subject']==s) &
                                              (df['SpeedCat']=='V5')]['Speed'].values[0]
# reorder columns and drop data for knee
df = df[['Subject', 'AgeGroup', 'Gender', 'Age', 'Height', 'Mass', 'BMI', 'LegLength',
         'SpeedCat', 'SpeedComf', 'Speed', 'StepLength', 'Cadence',
         'H2A_M', 'H2A_I', 'H2A_W']]
df

## Export df to csv
This file contains 51 subjects.

In [ ]:
df.to_csv(os.path.join('./../data','H2A_stats_input_51subs.csv'), index=False)